In [1]:
import confnotebook

In [2]:
from pathlib import Path

source = Path("../examples/test/full/")

files = sorted(source.glob("*.pdf"))

for i, file in enumerate(files):
    print(f"[{i}] {file.stem}")

[0] 10
[1] 126164
[2] 14964427_Енисейская ТГК-13-БРАЗ
[3] 14976087_АвеларСолар Тех-БРАЗ-1
[4] 15120979_Форвард Энерго-БРАЗ-1
[5] 15235008_ОГК-2-БРАЗ-1
[6] 25
[7] 33
[8] 4
[9] 44
[10] 7-1
[11] Акт сверки взаимных расчетов №00000379931 от 30.04.2024
[12] Акт сверки №0000
[13] Акт сверки №MOW00-0087974   от 10.06.2024
[14] Акт сверки №ТРБП-000006 от 10.01.2024
[15] АС ВНИИМ Менделеева Д.И. - БРАЗ на 31.12.24
[16] АС ВНИИМ Менделеева Д.И. - РУ на 31.12.24
[17] АС КРЕЗОЛ-САЗ на 31.08.25
[18] АС Охрана Металлург-САЗ на 31.12.25
[19] АС РУ- ВОСЬМОЙ ВЕТРОПАРК
[20] АС Фрейт Линк-БРАЗ на 30.09.25
[21] АСР СДД 2 кв.2024 (подп. к-а)
[22] Браз-Юнигрин Пауэр
[23] документ 23-ИИА-03-01141 от 31_03_2025
[24] документ 23-ИИА-03-01142 от 31_03_2025
[25] ЕВР-НКАЗ
[26] Неформализованный_первичный_документ_23_ИИА_03_00342_от_31_03
[27] Неформализованный_первичный_документ_23_ИИА_03_02596_от_31_03
[28] Неформализованный_первичный_документ_23_ИИА_03_03623_от_31_03
[29] Неформализованный_первичный_до

In [3]:
IDX_FILE = 17

In [4]:
from vision_core.debug_image_observer import DebugImageObserver

file = files[IDX_FILE]
# output_dir = f"../examples/output/{file.stem}"

# debug_image_observer = DebugImageObserver(output_dir=output_dir)

d:\projects\rusal_recon_srv\repo\recon_vision\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `DISABLE_MODEL_SOURCE_CHECK` to `True`.


In [5]:
from vision_core.pipelines.build_document import DocumentBuildPipeline

pipeline = DocumentBuildPipeline()

document = pipeline.build(file.read_bytes())

d:\projects\rusal_recon_srv\repo\recon_vision\.venv\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv5_server_det', 'D:\\projects\\rusal_recon_srv\\repo\\recon_vision\\models\\PP-OCRv5_server_det')
The specified device (GPU) is not available! Switching to CPU instead.
Creating model: ('cyrillic_PP-OCRv5_mobile_rec', 'D:\\projects\\rusal_recon_srv\\repo\\recon_vision\\models\\cyrillic_PP-OCRv5_mobile_rec')
The specified device (GPU) is not available! Switching to CPU instead.
2026-05-07 12:36:13.595 | INFO     | vision_core.pipelines.build_document:build:106 - Обработка страницы 0 с dpi 200...
2026-05-07 12:36:13.669 | INFO     | vision_core.pipelines.build_document:_process_page:189 - Коррекция ориентации и наклон

In [6]:
def get_cell_covering(table, row: int, col: int):
    for cell in table.get_rows()[row]:
        if cell.col <= col < cell.col + cell.colspan:
            return cell
    return None


summary_text = ""
summary_cell_text: list[str] = []
for page in document.pages:
    text_paragraph = " ".join(paragraph.text for paragraph in page.paragraphs)
    for table in page.tables:
        if table.continuation_of is not None:
            continue
        dc = table.dc_cols
        num_row = table.get_dc_header_row()
        if num_row == -1 or not dc:
            continue
        seen_cells: set[int] = set()
        for i, col in enumerate(sorted(dc)):
            for j in range(num_row):
                cell = get_cell_covering(table, j, col)
                if cell is None or id(cell) in seen_cells:
                    continue
                seen_cells.add(id(cell))
                cell_text = cell.value.strip()
                if cell_text:
                    summary_cell_text.append(cell_text)

    summary_text += text_paragraph + " "

print("SUMMARY CELL TEXT:")
print(summary_cell_text)
print("\nSUMMARY TEXT:")
print(summary_text)

SUMMARY CELL TEXT:
['по данным', 'ООО "ТД "Крезол"', 'АО "РУСАЛ\n1 Саяного', 'pck"']

SUMMARY TEXT:
Акт сверки взаимных расчетов № 2281 от 30 сентября 2025 г. за Январь 2025 г. - Август 2025 г. между ООО "ТД "Крезол" и АО "РУСАЛ Саяногорск" Мы, нижеподписавшиеся, Директор ООО "ТД "Крезол" Бахтиярова И. В., с одной стороны, и АО "РуСАЛ Саяногорск" , с другой стороны, составили настоящий акт сверки в том, что: 1. В период с 1 января 2025 г. по 31 августа 2025 г. были осуществлены следующие расчеты: Расчеты по объекту расчетов <не указан> в валюте RUB 2. Таким образом, на 31 августа 2025 г.: долг АO "РУСАЛ Саяногорск" в валюте RUВ 120 464,76 (Сто двадцать тысяч четыреста шестьдесят четыре рубля 76 копеек) О00 "ТД "Крезол", ИНН 0276162440, 450027, Республика Башкортостан, г Уфа, ул Трамвайная, д. 2/4, этаж 4 3. По результатам сверки расхождений не выявлено. АО "РУСАЛ Саяногорск", ИНН 1902014500, 655603, РФ, РХ, г.Саяногорск, территория Промплощадка Директор должность Gsp ТОРГОВЫЙДОМ о81 «К

In [7]:
import re

_LAT2CYR = str.maketrans({
    'A':'А','B':'В','C':'С','E':'Е','H':'Н','K':'К','M':'М','O':'О','P':'Р','T':'Т','X':'Х','Y':'У',
    'a':'а','c':'с','e':'е','o':'о','p':'р','x':'х','y':'у','k':'к','m':'м','h':'н','b':'в','t':'т',
    'R':'Р','r':'р','V':'В','v':'в'
})
_QUOTES = '«»\u201c\u201d\u201e\u2018\u2019\u201a\u2039\u203a'
_QUOTE_NORM = str.maketrans(_QUOTES, '"' * len(_QUOTES))

def normalize_text(text: str) -> str:
    if not text:
        return ""
    text = text.translate(_QUOTE_NORM)
    text = re.sub(r'"{2,}', '"', text)          # "" -> "
    text = re.sub(r'(\S)"', r'\1 "', text)      # ОБЩЕСТВО" -> ОБЩЕСТВО "
    text = text.translate(_LAT2CYR)
    text = text.upper().replace('Ё', 'Е')
    text = re.sub(r'\b000\b', 'ООО', text)
    text = re.sub(r'\s+', ' ', text, flags=re.UNICODE)
    return text.strip()



normalized_text = normalize_text(summary_text)

summary_cell_text_norm = [normalize_text(cell) for cell in summary_cell_text]


print("SUMMARY CELL TEXT:")
print(summary_cell_text_norm)
print("\nSUMMARY TEXT:")
print(normalized_text)

SUMMARY CELL TEXT:
['ПО ДАННЫМ', 'ООО "ТД "КРЕЗОЛ "', 'АО "РУСАЛ 1 САЯНОГО', 'РСК "']

SUMMARY TEXT:
АКТ СВЕРКИ ВЗАИМНЫХ РАСЧЕТОВ № 2281 ОТ 30 СЕНТЯБРЯ 2025 Г. ЗА ЯНВАРЬ 2025 Г. - АВГУСТ 2025 Г. МЕЖДУ ООО "ТД "КРЕЗОЛ " И АО "РУСАЛ САЯНОГОРСК " МЫ, НИЖЕПОДПИСАВШИЕСЯ, ДИРЕКТОР ООО "ТД "КРЕЗОЛ " БАХТИЯРОВА И. В., С ОДНОЙ СТОРОНЫ, И АО "РУСАЛ САЯНОГОРСК " , С ДРУГОЙ СТОРОНЫ, СОСТАВИЛИ НАСТОЯЩИЙ АКТ СВЕРКИ В ТОМ, ЧТО: 1. В ПЕРИОД С 1 ЯНВАРЯ 2025 Г. ПО 31 АВГУСТА 2025 Г. БЫЛИ ОСУЩЕСТВЛЕНЫ СЛЕДУЮЩИЕ РАСЧЕТЫ: РАСЧЕТЫ ПО ОБЪЕКТУ РАСЧЕТОВ <НЕ УКАЗАН> В ВАЛЮТЕ РUВ 2. ТАКИМ ОБРАЗОМ, НА 31 АВГУСТА 2025 Г.: ДОЛГ АО "РУСАЛ САЯНОГОРСК " В ВАЛЮТЕ РUВ 120 464,76 (СТО ДВАДЦАТЬ ТЫСЯЧ ЧЕТЫРЕСТА ШЕСТЬДЕСЯТ ЧЕТЫРЕ РУБЛЯ 76 КОПЕЕК) О00 "ТД "КРЕЗОЛ ", ИНН 0276162440, 450027, РЕСПУБЛИКА БАШКОРТОСТАН, Г УФА, УЛ ТРАМВАЙНАЯ, Д. 2/4, ЭТАЖ 4 3. ПО РЕЗУЛЬТАТАМ СВЕРКИ РАСХОЖДЕНИЙ НЕ ВЫЯВЛЕНО. АО "РУСАЛ САЯНОГОРСК ", ИНН 1902014500, 655603, РФ, РХ, Г.САЯНОГОРСК, ТЕРРИТОРИЯ ПРОМПЛОЩАДКА ДИРЕКТОР ДОЛЖНОСТЬ GSР ТОРГОВЫЙДО

In [8]:
ORGFORMS = {
    'АО':   'акционерное общество',
    'ОАО':  'открытое акционерное общество',
    'ЗАО':  'закрытое акционерное общество',
    'ООО':  'общество с ограниченной ответственностью',
    'ИП':   'индивидуальный предприниматель',
    'ПАО':  'публичное акционерное общество',
    'НП':   'некоммерческое партнерство',
    'ГУП':  'государственное унитарное предприятие',
    'МУП':  'муниципальное унитарное предприятие',
    'ФГУП': 'федеральное государственное унитарное предприятие',
}
ORGFORMS_FULL2SHORT = {v.upper(): k for k, v in ORGFORMS.items()}
def ocr_robust(s: str) -> str:
    """
    Заменяет букву "О" на паттерн, который может соответствовать как "О",
    так и "0", для повышения устойчивости к ошибкам OCR.
    """
    return s.replace('О', '[О0]').replace('о', '[о0]')

org_forms_pattern = "|".join(
    ocr_robust(k) for k in sorted(ORGFORMS.keys(), key=len, reverse=True)
) + r'|(?:' + "|".join(
    ocr_robust(v) for v in ORGFORMS.values()
) + r')'

In [9]:
def fix_rusal(name: str) -> str:
    """
    Корректирует специфические ошибки в написании названия "РУСАЛ"
    (например, "РУСАЛСАЯНОГОРСК" -> "РУСАЛ САЯНОГОРСК").
    """
    # Используем группу захвата для сохранения первой буквы после "РУСАЛ"
    return re.sub(r'РУСАЛ([А-ЯЁ])', r'РУСАЛ \1', name)

def extract_org_names(text: str) -> list[str]:
    """
    Извлекает уникальные полные названия организаций из заданного текста.

    Функция использует два прохода:
    1. Поиск по юридическим формам (якорь: АО, ООО и т.д.).
    2. Поиск названий, начинающихся с "РУСАЛ" без явной формы.

    Args:
        text: Текст, из которого необходимо извлечь названия организаций.

    Returns:
        list[str]: Список уникально найденных полных названий организаций.
    """
    results: list[str] = []
    seen_full: set[str] = set()
    seen_names: set[str] = set()

    # Регулярное выражение для поиска всех юридических форм
    org_re = re.compile(r'\b(?:' + org_forms_pattern + r')\b', re.IGNORECASE)

    # Проход 1: Поиск по юридическим формам (Якорь)
    for m in org_re.finditer(text):
        # Берем контекст 150 символов после найденной формы
        rest = text[m.end(): m.end() + 150]

        # Захватывает текст в кавычках, игнорируя вложенные
        q = re.search(r'"((?:[^"]*"(?=[А-ЯЁA-Za-zа-яё]))*[^"]*)"', rest)
        # ищем текст в кавычках, который может быть без пробелов
        if not q:
            q = re.search(r'"(\S+)', rest)

        if not q:
            continue

        name = fix_rusal(q.group(1).strip()).replace('"', '')
        matched = m.group().strip().replace('0', 'О').upper()
        short_form = ORGFORMS_FULL2SHORT.get(matched, matched)  # полная -> краткая, или уже краткая
        full_name = name + ', ' + short_form

        if full_name not in seen_full:
            seen_full.add(full_name)
            seen_names.add(name)
            results.append(full_name)

    # Проход 2: Поиск "РУСАЛ..." без орг.формы
    # Ищем любые кавычки, начинающиеся с "РУСАЛ", независимо от того, что следует за ними.
    # Паттерн r'"(РУСАЛ[^"]*)"' захватывает все, что в кавычках и начинается с РУСАЛ.
    for q in re.finditer(r'"(РУСАЛ[^"]*)"', text):
        name = fix_rusal(q.group(1).strip()).replace('"', '')
        if name not in seen_names:
            seen_names.add(name)
            results.append(name + ", ")

    return results


print("Из ячеек:", extract_org_names(" ".join(summary_cell_text_norm)))
print("\nИз полного текста:", extract_org_names(normalized_text))


Из ячеек: ['ТД КРЕЗОЛ, ООО', 'РУСАЛ 1 САЯНОГО РСК, АО']

Из полного текста: ['ТД КРЕЗОЛ, ООО', 'РУСАЛ САЯНОГОРСК, АО']


In [10]:
from enum import Enum


class Role(Enum):
    BUYER = 0
    SELLER = 1
    UNKNOWN = -1


ROLE_GLOSSARY = {
    r'\bОТ ПОКУПАТЕЛЯ\b': 'BUYER_SOURCE',
    r'\bОТ ПРОДАВЦА\b':   'SELLER_SOURCE',
    r'\bМЕЖДУ\b':         'PARTICIPATION_SCOPE',
}


def _org_token(full_name: str) -> str:
    return full_name.split(",", 1)[0].strip()


def _find_events(text: str) -> list[dict]:
    found = []
    for pattern, role in ROLE_GLOSSARY.items():
        for m in re.finditer(pattern, text, re.IGNORECASE):
            found.append({"keyword": m.group(), "role": role,
                          "start_index": m.start(), "end_index": m.end()})
    return sorted(found, key=lambda x: x["start_index"])


def _find_orgs_in_span(text: str, left: int, right: int, orgs: list[str]) -> list[str]:
    span = text[left:right]
    return [o for o in orgs if _org_token(o) and _org_token(o) in span]


def _find_working_pair(text: str, orgs: list[str], events: list[dict]) -> list[str]:
    """Два контрагента рядом с якорем МЕЖДУ."""
    anchor = next((e for e in events if e["role"] == "PARTICIPATION_SCOPE"), None)
    if anchor:
        window = text[anchor["end_index"]: anchor["end_index"] + 400]
        hits = sorted(
            [(window.find(_org_token(o)), o) for o in orgs if _org_token(o) in window]
        )
        pair = [o for _, o in hits[:2]]
        if len(pair) == 2:
            return pair
    return orgs[:2]


def _apply_symmetry(roles: dict) -> None:
    buyers   = [o for o, r in roles.items() if r == Role.BUYER]
    sellers  = [o for o, r in roles.items() if r == Role.SELLER]
    unknowns = [o for o, r in roles.items() if r == Role.UNKNOWN]
    if buyers and unknowns and not sellers:
        for o in unknowns:
            print(f"Правило симметрии: {o} -> SELLER")
            roles[o] = Role.SELLER
    elif sellers and unknowns and not buyers:
        for o in unknowns:
            print(f"Правило симметрии: {o} -> BUYER")
            roles[o] = Role.BUYER

def _deduplicate_orgs(orgs: list[str]) -> list[str]:
    tokens = [_org_token(o) for o in orgs]
    return [
        org for i, org in enumerate(orgs)
        if not any(tokens[i] in tokens[j] and tokens[i] != tokens[j] for j in range(len(tokens)))
    ]

def assign_roles(text: str, orgs: list[str]) -> dict[str, Role]:
    events = _find_events(text)
    print("Найдены события:", events)
    pair   = _find_working_pair(text, orgs, events)
    print("Рабочая пара:", pair)
    roles  = {o: Role.UNKNOWN for o in pair}

    # 1) РУСАЛ -> всегда BUYER (жёсткое правило)
    for o in pair:
        if "РУСАЛ" in _org_token(o):
            roles[o] = Role.BUYER
            print(f"Правило: {o} содержит 'РУСАЛ' -> BUYER")
    _apply_symmetry(roles)

    # 2) Явные якоря ОТ ПОКУПАТЕЛЯ / ОТ ПРОДАВЦА (только для UNKNOWN)
    for idx, event in enumerate(events):
        if event["role"] not in {"BUYER_SOURCE", "SELLER_SOURCE"}:
            continue
        next_start = events[idx + 1]["start_index"] if idx + 1 < len(events) else len(text)
        target = Role.BUYER if event["role"] == "BUYER_SOURCE" else Role.SELLER
        for o in _find_orgs_in_span(text, event["end_index"], next_start, pair):
            if roles[o] == Role.UNKNOWN:
                roles[o] = target
                print(f"Правило: {o} находится в диапазоне {event['keyword']} -> {target.name}")
    _apply_symmetry(roles)

    # 3) Позиционный фоллбек: первый -> SELLER, второй -> BUYER
    unknowns = [o for o, r in roles.items() if r == Role.UNKNOWN]
    if unknowns:
        roles[unknowns[0]] = Role.SELLER
        print(f"Правило позиционного фоллбека: {unknowns[0]} -> SELLER")
        for o in unknowns[1:]:
            roles[o] = Role.BUYER
            print(f"Правило позиционного фоллбека: {o} -> BUYER")

    return roles


# --- запуск ---
orgs_name = extract_org_names(" ".join(summary_cell_text_norm))
orgs_name_full = extract_org_names(normalized_text)

if len(orgs_name) >= 2:
    orgs_name_full_filtered = orgs_name
elif orgs_name:
    orgs_name_set = set(orgs_name)
    orgs_name_full_filtered = [
        o for o in orgs_name_full if any(n in o for n in orgs_name_set)
    ]
else:
    orgs_name_full_filtered = orgs_name_full

orgs_name_full_filtered = _deduplicate_orgs(orgs_name_full_filtered)
print("Организации (после дедупликации):", orgs_name_full_filtered)


roles_by_org = assign_roles(normalized_text, orgs_name_full_filtered)

for org, role in roles_by_org.items():
    print(f"- {org} -> {role.name}")

buyers  = [o for o, r in roles_by_org.items() if r == Role.BUYER]
sellers = [o for o, r in roles_by_org.items() if r == Role.SELLER]
print("\nBUYER:", buyers)
print("SELLER:", sellers)


Организации (после дедупликации): ['ТД КРЕЗОЛ, ООО', 'РУСАЛ 1 САЯНОГО РСК, АО']
Найдены события: [{'keyword': 'МЕЖДУ', 'role': 'PARTICIPATION_SCOPE', 'start_index': 94, 'end_index': 99}]
Рабочая пара: ['ТД КРЕЗОЛ, ООО', 'РУСАЛ 1 САЯНОГО РСК, АО']
Правило: РУСАЛ 1 САЯНОГО РСК, АО содержит 'РУСАЛ' -> BUYER
Правило симметрии: ТД КРЕЗОЛ, ООО -> SELLER
- ТД КРЕЗОЛ, ООО -> SELLER
- РУСАЛ 1 САЯНОГО РСК, АО -> BUYER

BUYER: ['РУСАЛ 1 САЯНОГО РСК, АО']
SELLER: ['ТД КРЕЗОЛ, ООО']
